In [1]:
# QA_QC tests for Tide Gauge data
import os
os.chdir("C:/Users/SHRINJANA/Downloads/Actual_Data")
import pandas as pd

In [109]:
df = pd.read_csv("tide_actual_wt.csv")

In [112]:
df.tail(10)

,OBJECTID,stationid,date,time,actual
2182470,2182470,wt,31/12/2021 0:00,0/1/1900 23:00,2.42
2182471,2182471,wt,31/12/2021 0:00,0/1/1900 23:06,2.40
2182472,2182472,wt,31/12/2021 0:00,0/1/1900 23:12,2.38
2182473,2182473,wt,31/12/2021 0:00,0/1/1900 23:18,2.34
2182474,2182474,wt,31/12/2021 0:00,0/1/1900 23:24,2.31
2182475,2182475,wt,31/12/2021 0:00,0/1/1900 23:30,2.28
2182476,2182476,wt,31/12/2021 0:00,0/1/1900 23:36,2.24
2182477,2182477,wt,31/12/2021 0:00,0/1/1900 23:42,2.20
2182478,2182478,wt,31/12/2021 0:00,0/1/1900 23:48,2.16
2182479,2182479,wt,31/12/2021 0:00,0/1/1900 23:54,2.11


In [47]:
data = {'Tn': [10, 10, 10, 10, 10, 10, 10, 12, 13, 14, 14, 14, 14, 14, 14, 14, None, None, None, None, None, None]}
df = pd.DataFrame(data)

# Define number of recent observations to check
REP_CNT_FAIL = 4

# Check if last REP_CNT_FAIL observations are equal
df['flag'] = 0  # Default flag value
df.loc[df['Tn'].rolling(REP_CNT_FAIL).apply(lambda x: len(set(x)) == 1, raw=True) == 1, 'flag'] = 4

print(df)

      Tn  flag
0   10.0     0
1   10.0     0
2   10.0     0
3   10.0     4
4   10.0     4
5   10.0     4
6   10.0     4
7   12.0     0
8   13.0     0
9   14.0     0
10  14.0     0
11  14.0     0
12  14.0     4
13  14.0     4
14  14.0     4
15  14.0     4
16   NaN     0
17   NaN     0
18   NaN     0
19   NaN     0
20   NaN     0
21   NaN     0


In [93]:
df['time'] = df['time'].str.split(' ').str[-1]

In [94]:
df['date'] = df['date'].str.split(' ').str[0]

In [95]:
df['date'] = df['date'] + ' ' + df['time']

In [99]:
df.tail(10)

,OBJECTID,stationid,date,time,actual
2197940,2197940,tm,31/12/2021 23:00:00,23:00:00,2.68
2197941,2197941,tm,31/12/2021 23:06:00,23:06:00,2.66
2197942,2197942,tm,31/12/2021 23:12:00,23:12:00,2.64
2197943,2197943,tm,31/12/2021 23:18:00,23:18:00,2.62
2197944,2197944,tm,31/12/2021 23:24:00,23:24:00,2.59
2197945,2197945,tm,31/12/2021 23:30:00,23:30:00,2.57
2197946,2197946,tm,31/12/2021 23:36:00,23:36:00,2.54
2197947,2197947,tm,31/12/2021 23:42:00,23:42:00,2.51
2197948,2197948,tm,31/12/2021 23:48:00,23:48:00,2.49
2197949,2197949,tm,31/12/2021 23:54:00,23:54:00,2.46


In [98]:
df.to_csv("processed_data_tm_test.csv", index=False)

In [ ]:
df['time'][1753944]

'3:48:00'

In [74]:
df1 = pd.read_csv("tide_actual_wt.csv")

In [75]:
df1['time'][1753944]

'0/1/1900 10:24'

In [59]:
df.tail(5)

,OBJECTID,stationid,date,time,actual
2202461,2202461,tp,31/12/2021 0:00,23:30:00,2.66
2202462,2202462,tp,31/12/2021 0:00,23:36:00,2.63
2202463,2202463,tp,31/12/2021 0:00,23:42:00,2.61
2202464,2202464,tp,31/12/2021 0:00,23:48:00,2.58
2202465,2202465,tp,31/12/2021 0:00,23:54:00,2.55


In [100]:
df = pd.read_csv("missing_times_WT.csv")

In [77]:
df['date'][1753944]

'NaT'

In [62]:
import numpy as np

#df['actual'] = df['actual'].replace(0, np.nan)

# Step 1: Flag out-of-range values (example: assuming valid range is 0 to 10)
df['flag_out_of_range'] = (df['actual'] < 0) | (df['actual'] > 4)

# Step 2: Flag stuck or oscillating values (example: if no change for 5 consecutive readings

# Check if last REP_CNT_FAIL observations are equal
df['flag_stuck'] = 1  # Default flag value
df.loc[df['actual'].rolling(5).apply(lambda x: len(set(x)) == 1, raw=True) == 1, 'flag_stuck'] = 0


# Step 3: Spike detection (example: large change compared to previous values)
df['diff'] = df['actual'].diff().abs()
df['flag_spike'] = df['diff'] > (df['diff'].mean() + 3 * df['diff'].std())

# Step 4: Remove flagged values and calculate simple hourly means
df['flag_count'] = df[['flag_stuck', 'flag_spike']].sum(axis=1)

df.loc[df['flag_count'] >= 2, 'actual'] = np.nan

# Remove rows with 2 or more flags
#df_filtered = df[df['flag_count'] < 2]

# Drop the 'flag_count' column, as it's no longer needed
#df_filtered = df_filtered.drop(columns=['flag_count', 'flag_spike', 'diff', 'flag_out_of_range', 'flag_stuck'])



In [63]:
df.head(10)

,date,actual,flag_out_of_range,flag_stuck,diff,flag_spike,flag_count
0,01-Jan-1996 00:00:00,1.78,False,1,NaN,False,1
1,01-Jan-1996 00:06:00,NaN,False,1,NaN,False,1
2,01-Jan-1996 00:12:00,NaN,False,1,NaN,False,1
3,01-Jan-1996 00:18:00,NaN,False,1,NaN,False,1
4,01-Jan-1996 00:24:00,NaN,False,1,NaN,False,1
5,01-Jan-1996 00:30:00,NaN,False,1,NaN,False,1
6,01-Jan-1996 00:36:00,NaN,False,1,NaN,False,1
7,01-Jan-1996 00:42:00,NaN,False,1,NaN,False,1
8,01-Jan-1996 00:48:00,NaN,False,1,NaN,False,1
9,01-Jan-1996 00:54:00,NaN,False,1,NaN,False,1


In [53]:
df = df.drop(columns=['flag_count', 'flag_spike', 'diff', 'flag_out_of_range', 'flag_stuck'])

In [43]:
df_filtered

,date,actual
0,24-Nov-1996 00:06:00,2.49
1,24-Nov-1996 00:12:00,2.46
2,24-Nov-1996 00:18:00,2.42
3,24-Nov-1996 00:24:00,2.38
4,24-Nov-1996 00:30:00,2.34
...,...,...
2200554,31-Dec-2021 23:30:00,2.28
2200555,31-Dec-2021 23:36:00,2.24
2200556,31-Dec-2021 23:42:00,2.20
2200557,31-Dec-2021 23:48:00,2.16


In [55]:
df.to_csv("processed_data_wt_test_5.csv", index=False)

In [56]:
df

,date,actual
0,24-Nov-1996 00:06:00,2.49
1,24-Nov-1996 00:12:00,2.46
2,24-Nov-1996 00:18:00,2.42
3,24-Nov-1996 00:24:00,2.38
4,24-Nov-1996 00:30:00,2.34
...,...,...
2200554,31-Dec-2021 23:30:00,2.28
2200555,31-Dec-2021 23:36:00,2.24
2200556,31-Dec-2021 23:42:00,2.20
2200557,31-Dec-2021 23:48:00,2.16


In [13]:
df['not_nan'] = df['actual'].notna()

# Step 2: Create a new column to label continuous blocks of non-NaN values
df['group'] = (df['not_nan'] != df['not_nan'].shift()).cumsum()

# Step 3: Filter out groups with NaN values and find the longest block
non_nan_groups = df[df['not_nan']].groupby('group').size()

# Step 4: Identify the group with the longest continuous block
longest_group = non_nan_groups.idxmax()

# Step 5: Extract the rows for the longest block
longest_block = df[df['group'] == longest_group]

In [9]:
longest_block.to_csv("processed_data_wt_test_longest.csv", index=False)

In [10]:
longest_block

,OBJECTID,stationid,date,time,actual,flag_out_of_range,flag_stuck,diff,flag_spike,not_nan,group
1367074,1367074,wt,16/8/2012 0:00,1899-12-30 11:24,2.37,False,False,0.01,False,True,11319
1367075,1367075,wt,16/8/2012 0:00,1899-12-30 11:30,2.37,False,False,0.00,False,True,11319
1367076,1367076,wt,16/8/2012 0:00,1899-12-30 11:36,2.36,False,False,0.01,False,True,11319
1367077,1367077,wt,16/8/2012 0:00,1899-12-30 11:42,2.34,False,False,0.02,False,True,11319
1367078,1367078,wt,16/8/2012 0:00,1899-12-30 11:48,2.33,False,False,0.01,False,True,11319
...,...,...,...,...,...,...,...,...,...,...,...
1370181,1370181,wt,29/8/2012 0:00,1899-12-30 10:06,2.31,False,False,0.01,False,True,11319
1370182,1370182,wt,29/8/2012 0:00,1899-12-30 10:12,2.31,False,False,0.00,False,True,11319
1370183,1370183,wt,29/8/2012 0:00,1899-12-30 10:18,2.31,False,False,0.00,False,True,11319
1370184,1370184,wt,29/8/2012 0:00,1899-12-30 10:24,2.31,False,False,0.00,False,True,11319


In [11]:
print("Start of block:", longest_block.iloc[0]['date'])
print("End of block:", longest_block.iloc[-1]['date'])

Start of block: 16/8/2012 0:00
End of block: 29/8/2012 0:00
